<a href="https://colab.research.google.com/github/Eton-np/DataMining/blob/main/Chapter_1_Worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ใบงานปฏิบัติการ สัปดาห์ที่ 1
## 128-445 การทำคลังข้อมูลและเหมืองข้อมูล

**ชื่อ–สกุล:** Test  **รหัสนักศึกษา:** ....................  **กลุ่ม:** ..........

---

### สิ่งที่จะได้ทำในใบงานนี้

| ปฏิบัติการ | หัวข้อ | เวลา |
|---|---|---|
| **1** | สำรวจข้อมูลจริง แยก Fact ออกจาก Dimension และทำ OLAP | 20 นาที |
| **2** | สหสัมพันธ์ไม่ใช่เหตุและผล และกับดัก Data Leakage | 8 นาที |
| **3** | จับผิด AI ด้วยโค้ดของตัวเอง | 15 นาที |

### กติกา

1. **เขียนโค้ดเอง** ทุกช่องที่เขียนว่า `# TODO`
2. **ห้ามเดา** — ทุกคำตอบต้องมีโค้ดที่พิสูจน์ได้
3. ใช้ AI ช่วยได้ แต่ต้อง **รันเองและตรวจสอบผลเอง** และระบุไว้ท้ายใบงาน
4. เซลล์ที่ขึ้นต้นด้วย `ck(...)` คือเซลล์ตรวจคำตอบอัตโนมัติ — รันเพื่อดูว่าถูกหรือยัง

---
## ขั้นที่ 0 · เตรียมข้อมูล

รันเซลล์ข้างล่างนี้ก่อนเสมอ ถ้าเครือข่ายของมหาวิทยาลัยบล็อก GitHub ให้อัปโหลดไฟล์ `supermarket_sales.csv` เข้า Colab แล้วรันใหม่ — โค้ดจะสลับไปอ่านไฟล์ในเครื่องให้เอง

In [ ]:
import pandas as pd
import numpy as np

URL = ('https://raw.githubusercontent.com/plotly/datasets/master/'
       'supermarket_Sales.csv')
try:
    df = pd.read_csv(URL)
    print("โหลดจากอินเทอร์เน็ตสำเร็จ")
except Exception:
    df = pd.read_csv('supermarket_sales.csv')
    print("โหลดจากไฟล์ในเครื่องสำเร็จ")

# ทำให้ชื่อคอลัมน์เป็นมาตรฐานเดียวกัน (ตัวเล็ก ไม่มีเว้นวรรค)
df.columns = (df.columns.str.strip().str.lower()
                .str.replace(' ', '_').str.replace('%', ''))
df = df.rename(columns={'customer_stratification_rating': 'rating',
                        'cost_of_goods_sold': 'cogs',
                        'gross_margin_percentage': 'gross_margin_pct'})
df.head(3)

In [ ]:
# ===== เซลล์ตรวจคำตอบ — ห้ามแก้ =====
import base64
_A = {"rows": "MTAwMA==", "cols": "MTc=", "missing": "MA==", "dup": "MA==", "branch_C": "MTEwNTY4Ljcx", "corr_rt": "LTAuMDM2", "gap_pct": "NjEuMzI=", "gmp_nunique": "MQ=="}

def ck(key, value, hint=""):
    """ตรวจคำตอบ (เฉลยถูกเข้ารหัสไว้ เพื่อไม่ให้อ่านเจอโดยบังเอิญ)"""
    exp = base64.b64decode(_A[key]).decode()
    try:
        tol = max(0.02, abs(float(exp)) * 1e-5)
        ok = abs(float(value) - float(exp)) <= tol
    except (TypeError, ValueError):
        ok = str(value).strip() == exp
    print(("[ถูกต้อง]   " if ok else "[ยังไม่ถูก] ") + key + " = " + str(value)
          + ("" if ok else "   " + hint))
    return ok

print("โหลดตัวตรวจคำตอบเรียบร้อย")

---
# ปฏิบัติการที่ 1 · สำรวจข้อมูลจริงและทำ OLAP
**เวลา 20 นาที · ทำเป็นกลุ่ม 4 คน**

> เป้าหมาย: แตะข้อมูลจริงเป็นครั้งแรก และพิสูจน์ด้วยตัวเองว่า **ยอดรวมปิดบังความจริงได้อย่างไร**

### 1.1 ข้อมูลชุดนี้ใหญ่แค่ไหน

In [ ]:
# TODO: หาจำนวนแถวและจำนวนคอลัมน์ เก็บไว้ในตัวแปร n_rows และ n_cols
n_rows = ...
n_cols = ...

print(n_rows, n_cols)

In [ ]:
ck("rows", n_rows, "ลองใช้ df.shape")
ck("cols", n_cols, "ลองใช้ df.shape")

### 1.2 มีค่าสูญหายหรือแถวซ้ำไหม

ก่อนเชื่อผลลัพธ์ใด ๆ ต้องรู้ก่อนว่าข้อมูลสะอาดแค่ไหน

In [ ]:
# TODO: นับจำนวนช่องที่ว่างทั้งหมด และจำนวนแถวที่ซ้ำกันทั้งแถว
# คำใบ้: df.isnull().sum().sum()  และ  df.duplicated().sum()
n_missing = ...
n_dup = ...

print("ช่องที่ว่างทั้งหมด:", n_missing)
print("แถวที่ซ้ำกันทั้งแถว:", n_dup)

In [ ]:
ck("missing", n_missing)
ck("dup", n_dup)

### 1.3 คอลัมน์ไหนเป็น Fact คอลัมน์ไหนเป็น Dimension

**เคล็ดลับ:** จำนวนค่าไม่ซ้ำ (`nunique`) คือเบาะแสที่เร็วที่สุด
- ค่าไม่ซ้ำ **น้อย** → มักเป็น **Dimension**
- ค่าไม่ซ้ำ **เยอะ และเป็นตัวเลขต่อเนื่อง** → มักเป็น **Fact**
- ค่าไม่ซ้ำ **เท่ากับ 1** → คอลัมน์ที่ไม่ให้สารสนเทศใด ๆ เลย

In [ ]:
# TODO: แสดงจำนวนค่าไม่ซ้ำของทุกคอลัมน์ เรียงจากน้อยไปมาก
# คำใบ้: df.nunique() แล้วต่อด้วย .sort_values()


In [ ]:
# TODO: คอลัมน์ใดที่มีค่าไม่ซ้ำเพียง 1 ค่า
#       เก็บจำนวนค่าไม่ซ้ำของคอลัมน์นั้นไว้ในตัวแปร gmp_nunique
gmp_nunique = ...

print("gross_margin_pct มีค่าไม่ซ้ำ:", gmp_nunique, "ค่า")

In [ ]:
ck("gmp_nunique", gmp_nunique, "ดูคอลัมน์ที่อยู่บนสุดของผลลัพธ์ข้อที่แล้ว")

**เขียนตอบ (ดับเบิลคลิกเพื่อพิมพ์):**

| คอลัมน์ | จำนวนค่าไม่ซ้ำ | Fact / Dimension / ตัดทิ้ง | เหตุผล |
|---|---|---|---|
| `branch` | | | |
| `product_line` | | | |
| `unit_price` | | | |
| `total` | | | |
| `gross_margin_pct` | | | |

**คำถาม:** ทำไมคอลัมน์ที่มีค่าไม่ซ้ำเพียง 1 ค่า ถึงไม่มีประโยชน์กับการวิเคราะห์

*ตอบ:* ...........................................................................................................

### 1.4 Roll-up — ยอดขายรวมของแต่ละสาขา

นี่คือท่า OLAP ท่าแรก: สรุปข้อมูลขึ้นไปที่ระดับหยาบกว่า

In [ ]:
# TODO: หายอดขายรวม (total) ของแต่ละสาขา (branch) ปัดทศนิยม 2 ตำแหน่ง
#       เก็บผลไว้ในตัวแปร branch_total
branch_total = ...

print(branch_total)
print()
print("ต่างกันมากที่สุด %.2f%%" %
      (100 * (branch_total.max() - branch_total.min()) / branch_total.min()))

In [ ]:
ck("branch_C", branch_total['C'], "ลองใช้ df.groupby('branch')['total'].sum()")

**หยุดคิดก่อนไปต่อ:** ยอดขายรวมของทั้ง 3 สาขาต่างกันไม่ถึง 4%

> ถ้าคุณเป็นผู้บริหารและเห็นแค่ตัวเลขชุดนี้ คุณจะตัดสินใจอะไร
>
> *ตอบ:* .........................................................................................................

### 1.5 Dice — หั่นสองมิติพร้อมกัน

ท่านี้จะเปลี่ยนคำตอบของคุณไปโดยสิ้นเชิง

In [ ]:
# TODO: สร้างตาราง pivot ที่มี
#       แถว   = product_line
#       คอลัมน์ = branch
#       ค่า    = ผลรวมของ total
#       เก็บไว้ในตัวแปร p แล้วปัดเป็นจำนวนเต็ม
p = ...

print(p)

In [ ]:
# TODO: หาว่ากลุ่มสินค้าใด มีช่องว่างระหว่างสาขาที่ขายดีที่สุดกับแย่ที่สุด "มากที่สุด"
#       ให้คิดเป็น "เปอร์เซ็นต์" ไม่ใช่ผลต่างดิบ  ปัดทศนิยม 2 ตำแหน่ง
#       คำใบ้: (p.max(axis=1) / p.min(axis=1) - 1) * 100
gap_pct = ...
max_gap_pct = ...

print(gap_pct)
print("กลุ่มที่ต่างกันมากที่สุดคือ:", gap_pct.idxmax(), "ต่างกัน", max_gap_pct, "%")

In [ ]:
ck("gap_pct", max_gap_pct, "คิดเป็นเปอร์เซ็นต์ ไม่ใช่ผลต่างดิบ  ปัดทศนิยม 2 ตำแหน่ง")

**ข้อควรระวังที่สำคัญ — อภิปรายในกลุ่มหลังได้คำตอบแล้ว:**

ลองเทียบสองวิธีวัดนี้ดู

| วิธีวัด | สูตร | กลุ่มสินค้าที่ได้อันดับ 1 |
|---|---|---|
| ผลต่างดิบ | `p.max(axis=1) - p.min(axis=1)` | .......................... |
| เปอร์เซ็นต์ | `(p.max(axis=1) / p.min(axis=1) - 1) * 100` | .......................... |

> **สองวิธีนี้ให้คำตอบคนละกลุ่มสินค้า** — วิธีวัดที่ต่างกันให้คำตอบที่ต่างกัน
> เวลารายงานผล ต้องบอกให้ชัดเสมอว่ากำลังวัดด้วยอะไร มิฉะนั้นผู้ฟังจะเข้าใจผิด

### 1.6 วาดกราฟให้เห็นภาพ

รันเซลล์ข้างล่างได้เลย ไม่ต้องแก้

In [ ]:
import matplotlib.pyplot as plt

ax = p.plot(kind='barh', figsize=(9, 4.2), width=0.78,
            color=['#2E77B8', '#1E9C9C', '#E8A33D'])
ax.set_xlabel('Total sales'); ax.set_ylabel('')
ax.set_title('Sales by product line and branch', loc='left', fontsize=12)
ax.legend(title='Branch', frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', color='#D8E1EA', linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

### สรุปปฏิบัติการที่ 1 — เขียนตอบ

1. ยอดรวมของ 3 สาขาต่างกันไม่ถึง 4% แต่พอหั่นสองมิติแล้วต่างกันสูงสุดกี่เปอร์เซ็นต์

   *ตอบ:* ..............................................................................................

2. ถ้าคุณเป็นผู้จัดการสาขา C คุณจะทำอะไรต่อกับกลุ่ม Home and lifestyle และคุณต้องการข้อมูลอะไรเพิ่ม

   *ตอบ:* ..............................................................................................

---
# ปฏิบัติการที่ 2 · สหสัมพันธ์ไม่ใช่เหตุและผล
**เวลา 8 นาที**

> เป้าหมาย: ทดสอบสมมติฐานที่ *ฟังดูสมเหตุสมผลที่สุด* ด้วยโค้ด แล้วดูว่าสัญชาตญาณของเราเชื่อถือได้แค่ไหน

### 2.1 เดาก่อน แล้วค่อยพิสูจน์

**สมมติฐาน:** ลูกค้าที่ซื้อของมาก น่าจะพอใจมากกว่า จึงให้คะแนน `rating` สูงกว่า

**กลุ่มของคุณเดาว่าค่าสหสัมพันธ์จะเป็นเท่าไร (ระหว่าง −1 ถึง 1):** ..............

In [ ]:
# TODO: คำนวณค่าสหสัมพันธ์ระหว่าง total กับ rating ปัดทศนิยม 3 ตำแหน่ง
corr_rt = ...

print("corr(total, rating) =", corr_rt)

In [ ]:
ck("corr_rt", corr_rt, "ลองใช้ df['total'].corr(df['rating'])")

### 2.2 มีสมการซ่อนอยู่ในข้อมูลหรือเปล่า

In [ ]:
# TODO: ตรวจว่า total เท่ากับ cogs บวก tax_5 ทุกแถวหรือไม่
#       คำใบ้: np.allclose(a, b)
same = ...
print("total = cogs + tax_5 ทุกแถวหรือไม่ :", same)

# TODO: แสดงเมทริกซ์สหสัมพันธ์ของ unit_price, quantity, total, cogs, rating


### 2.3 อภิปรายในกลุ่ม — เขียนตอบ

1. ค่าสหสัมพันธ์ที่ได้จริง ต่างจากที่กลุ่มเดาไว้แค่ไหน และแปลว่าอะไร

   *ตอบ:* ..............................................................................................

2. `corr(cogs, total)` เท่ากับเท่าไร และทำไมถึงเป็นเช่นนั้น

   *ตอบ:* ..............................................................................................

3. ถ้าสร้างโมเดลทำนาย `total` แล้วใส่ `cogs` เข้าไปด้วย โมเดลจะแม่นแค่ไหน และ **โมเดลได้เรียนรู้อะไรจริง ๆ หรือไม่**

   *ตอบ:* ..............................................................................................

> ปรากฏการณ์ในข้อ 3 เรียกว่า **Data Leakage** — จะเรียนเต็ม ๆ ในสัปดาห์ที่ 11

---
# ปฏิบัติการที่ 3 · จับผิด AI ด้วยโค้ดของตัวเอง
**เวลา 15 นาที · ทำเป็นกลุ่ม**

อาจารย์ให้ AI สรุปชุดข้อมูลนี้ โดยบอก AI **แค่ชื่อคอลัมน์** ไม่ได้ให้ข้อมูลจริง
AI ตอบกลับมา 8 ข้อ ด้วยน้ำเสียงมั่นใจเท่ากันทุกข้อ

| # | ข้อความที่ AI สรุปมา |
|---|---|
| 1 | ชุดข้อมูลนี้มีค่าสูญหายจำนวนมากในคอลัมน์ `rating` |
| 2 | มีแถวที่ซ้ำกันประมาณ 3% ควรลบออกก่อนวิเคราะห์ |
| 3 | `Health and beauty` เป็นกลุ่มสินค้าที่ทำยอดขายสูงสุด |
| 4 | ยอดขายพุ่งสูงสุดในช่วงพักเที่ยงเวลา 12:00 น. |
| 5 | ลูกค้า `Member` ใช้จ่ายเฉลี่ยต่อบิลสูงกว่า `Normal` |
| 6 | ยอดขายรวมของ 3 สาขาต่างกันไม่ถึง 5% |
| 7 | คะแนน `rating` เฉลี่ยอยู่ที่ประมาณ 7 คะแนน |
| 8 | ลูกค้าเพศหญิงมียอดใช้จ่ายรวมสูงกว่าเพศชาย |

### ขั้นที่ 1 — เดาก่อน (ยังไม่ต้องรันโค้ดใด ๆ)

แก้ค่าในเซลล์ข้างล่างเป็น `True` (คิดว่าจริง) หรือ `False` (คิดว่าเท็จ)

In [ ]:
# TODO: เดาว่าแต่ละข้อจริงหรือเท็จ ก่อนรันโค้ดพิสูจน์
GUESS = {
    1: None,   # True หรือ False
    2: None,
    3: None,
    4: None,
    5: None,
    6: None,
    7: None,
    8: None,
}
print("บันทึกคำเดาแล้ว")

### ขั้นที่ 2 — พิสูจน์ทีละข้อด้วยโค้ด

In [ ]:
# TODO ข้อ 1: มีค่าสูญหายในคอลัมน์ rating กี่ค่า
# TODO ข้อ 2: มีแถวที่ซ้ำกันทั้งแถวกี่แถว


In [ ]:
# TODO ข้อ 3: ยอดขายรวมของแต่ละกลุ่มสินค้า เรียงจากน้อยไปมาก


In [ ]:
# TODO ข้อ 4: ชั่วโมงใดของวันที่มียอดขายรวมสูงสุด
# คำใบ้: สร้างคอลัมน์ hour ก่อน
#   df['hour'] = pd.to_datetime(df['time'], format='%H:%M').dt.hour


In [ ]:
# TODO ข้อ 5: ยอดใช้จ่ายเฉลี่ยต่อบิล ของ Member เทียบกับ Normal
# TODO ข้อ 6: ยอดขายรวมของแต่ละสาขา
# TODO ข้อ 7: คะแนน rating เฉลี่ย
# TODO ข้อ 8: ยอดใช้จ่ายรวม แยกตามเพศ


### ขั้นที่ 3 — บันทึกหลักฐานที่ได้

กรอกตัวเลขที่พิสูจน์ได้จริงลงในเซลล์ข้างล่าง (เป็นข้อความสั้น ๆ ก็ได้) แล้วจึงรันเซลล์ตรวจคำตอบ

In [ ]:
# TODO: กรอกตัวเลขที่พิสูจน์ได้จริงของแต่ละข้อ
EVIDENCE = {
    1: "",
    2: "",
    3: "",
    4: "",
    5: "",
    6: "",
    7: "",
    8: "",
}
print("บันทึกหลักฐานแล้ว")

In [ ]:
# ===== เซลล์เฉลย — รันหลังกรอก GUESS และ EVIDENCE ครบแล้วเท่านั้น =====
TRUTH = {1: False, 2: False, 3: False, 4: False,
         5: True, 6: True, 7: True, 8: True}
FACT = {
    1: "ไม่มีค่าสูญหายเลยแม้แต่ช่องเดียว",
    2: "แถวซ้ำ 0 แถว",
    3: "Health and beauty ต่ำสุด 49,193.74 — สูงสุดคือ Food and beverages 56,144.84",
    4: "สูงสุดคือ 19:00 น. (39,700) ส่วน 12:00 น. อยู่ที่ 26,066",
    5: "Member 327.79 · Normal 318.12",
    6: "C 110,568.71 · A 106,200.37 · B 106,197.67 (ต่างกันไม่ถึง 4%)",
    7: "6.97",
    8: "หญิง 167,882.92 · ชาย 155,083.82",
}

if any(v is None for v in GUESS.values()):
    print("ยังกรอก GUESS ไม่ครบ — กลับไปเดาให้ครบ 8 ข้อก่อน")
elif any(not str(v).strip() for v in EVIDENCE.values()):
    print("ยังกรอก EVIDENCE ไม่ครบ — ต้องพิสูจน์ด้วยโค้ดก่อนดูเฉลย")
else:
    score = 0
    for i in range(1, 9):
        mark = "ถูก " if GUESS[i] == TRUTH[i] else "ผิด "
        score += GUESS[i] == TRUTH[i]
        print("ข้อ %d  เดาว่า %-5s | ความจริง %-5s | %s | %s"
              % (i, GUESS[i], TRUTH[i], mark, FACT[i]))
    print()
    print("กลุ่มของคุณเดาถูก %d จาก 8 ข้อ" % score)
    print("AI ตอบผิด 4 ข้อจาก 8 ข้อ โดยใช้น้ำเสียงมั่นใจเท่ากันหมด")

### สรุปปฏิบัติการที่ 3 — เขียนตอบ

1. กลุ่มของคุณเดาถูกกี่ข้อจาก 8 ข้อ

   *ตอบ:* ..........

2. ข้อที่ AI ตอบผิดทั้ง 4 ข้อ **มีอะไรร่วมกัน**

   *ตอบ:* ..............................................................................................

3. ถ้าคุณไม่มีเวลาตรวจสอบเอง คุณจะแยกข้อที่ถูกออกจากข้อที่ผิดได้อย่างไร

   *ตอบ:* ..............................................................................................

---
# สรุปท้ายใบงาน

### เขียนตอบ 3 ข้อ

1. สิ่งที่ค้นพบวันนี้ที่ทำให้คุณ **แปลกใจที่สุด** คืออะไร เพราะอะไร

   *ตอบ:* ..............................................................................................

2. ยกตัวอย่างคำถามธุรกิจ 1 ข้อ ที่ข้อมูลชุดนี้ **ตอบไม่ได้** และบอกว่าต้องมีข้อมูลอะไรเพิ่ม

   *ตอบ:* ..............................................................................................

3. ถ้าจะทำโครงงาน Capstone จากข้อมูลชุดนี้ คุณจะตั้งคำถามว่าอะไร

   *ตอบ:* ..............................................................................................

---

### การใช้ AI ในใบงานนี้ (ตอบตามจริง)

| ข้อ | คำตอบ |
|---|---|
| ใช้เครื่องมือ AI ใดบ้าง | .............................................. |
| ใช้ช่วยในส่วนใด | .............................................. |
| ตรวจสอบผลลัพธ์อย่างไร | .............................................. |
| ไม่ได้ใช้เลย | ☐ |

> **หลักของรายวิชา:** AI เป็นผู้ช่วยที่เก่งแต่ไม่รับผิดชอบ — คนที่รับผิดชอบผลงานคือคุณ

---

### รายการตรวจก่อนส่ง

- [ ] รันทุกเซลล์ตั้งแต่ต้นจนจบแล้ว ไม่มี error
- [ ] เซลล์ `ck(...)` ทุกอันขึ้นว่า **[ถูกต้อง]**
- [ ] ตอบคำถามเขียนตอบครบทุกข้อ
- [ ] กรอกตารางการใช้ AI แล้ว
- [ ] เปลี่ยนชื่อไฟล์เป็น `W1_รหัสนักศึกษา_ชื่อ.ipynb` แล้วส่งใน Google Classroom

**กำหนดส่ง:** วันอาทิตย์ เวลา 23.59 น.